# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# constants
MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

MODEL_CHOICES = [
    ("GPT-4o-mini (OpenRouter)", MODEL_GPT, "openrouter"),
    ("Llama3.2 (Ollama)", MODEL_LLAMA, "ollama"),
]

# load env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# OpenRouter client
openrouter = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

# Ollama client
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# system prompt
SYSTEM_PROMPT = """
Act as a senior software engineer proficient in multiple programming languages.

Explain technical questions clearly using step-by-step explanations.
Use examples when helpful.
Assume the user is learning.
"""

# streaming function
def stream_reply(history, user_message, model_label):

    label_map = {label: (model, backend) for label, model, backend in MODEL_CHOICES}
    model_id, backend = label_map.get(model_label)

    client = openrouter if backend == "openrouter" else ollama

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant or ""})

    messages.append({"role": "user", "content": user_message})

    stream = client.chat.completions.create(
        model=model_id if backend=="ollama" else f"openai/{model_id}",
        messages=messages,
        stream=True
    )

    for chunk in stream:
        part = chunk.choices[0].delta.content or ""
        if part:
            yield part

# chat handler
def chat(message, history, model_choice):

    full_response = ""

    for chunk in stream_reply(history, message, model_choice):
        full_response += chunk
        yield full_response

# model selector
model_dropdown = gr.Dropdown(
    choices=[label for label, _, _ in MODEL_CHOICES],
    value=MODEL_CHOICES[0][0],
    label="Model"
)

# UI
demo = gr.ChatInterface(
    chat,
    additional_inputs=[model_dropdown],
    title="Technical Question Answering Tutor",
    description="Ask a coding question and get a step-by-step explanation.",
)

demo.launch()